# pop RAG sweep -- one-notebook Colab run (resumable)

Runs the **full 8-config RAG sweep** (BM25 / CodeBERT retrievers x k=0/1/3/5 exemplars,
prompting Qwen2.5-Coder-1.5B-Instruct over the CodeXGLUE Java refinement test split) via
`scripts/run_rag.py`, with every artifact stored on **your Google Drive** so nothing is
lost when a Colab session ends. k=0 is the zero-shot baseline.

**Before first run** (see `docs/colab-runbook.md` for details): create the shared Drive folder
`MyDrive/pop_cycle3/` and upload `pop_repo.zip` into it. The RAG, scaling and LoRA notebooks
share this one workspace, so the zip is uploaded here just once and every arm's
results co-locate under a single `results/` for aggregation. Build the zip locally from your
clone of https://github.com/yib7/pretrain-or-prompt with
`git archive --format=zip -o dist/pop_repo.zip HEAD` (the repo travels to Colab as a
Drive zip rather than a git clone, so no token is needed).

**Every session**: Runtime > Change runtime type > pick a GPU, then Runtime > **Run all**.
Safe to re-run any time -- completed configs are skipped (each `rag`/`eval` step has a durable
done-marker), and progress is always visible in
`Drive/pop_cycle3/logs/rag/sweep/STATUS.md`.

In [ ]:
import sys

print("Python", sys.version)
assert sys.version_info >= (3, 11), "pop needs Python >= 3.11; this Colab runtime is older"
!nvidia-smi

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/pop_cycle3")
for sub in ("outputs", "results", "logs"):
    (BASE / sub).mkdir(parents=True, exist_ok=True)

ZIP = BASE / "pop_repo.zip"
assert ZIP.is_file(), (
    f"Upload pop_repo.zip to {ZIP.parent} first -- build it locally with\n"
    "  git archive --format=zip -o dist/pop_repo.zip HEAD\n"
    "then drag dist/pop_repo.zip into Drive/pop_cycle3/ (see docs/colab-runbook.md)."
)
print("Drive workspace ready:", BASE)

In [ ]:
%cd /content
!rm -rf /content/repo
!unzip -q /content/drive/MyDrive/pop_cycle3/pop_repo.zip -d /content/repo
%cd /content/repo
%pip install -q -e .

In [ ]:
# Point the repo's outputs/, results/, logs/ at Drive so predictions, metrics, and
# progress logs all survive session resets. The repo ships a committed results/
# directory; its contents are copied onto Drive once before the swap.
import os
import shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/pop_cycle3")
REPO = Path("/content/repo")
for sub in ("outputs", "results", "logs"):
    drive_dir = BASE / sub
    repo_dir = REPO / sub
    if repo_dir.is_symlink():
        repo_dir.unlink()
    elif repo_dir.exists():
        shutil.copytree(repo_dir, drive_dir, dirs_exist_ok=True)
        shutil.rmtree(repo_dir)
    os.symlink(drive_dir, repo_dir)
print("outputs/, results/, logs/ now live on Drive:", BASE)

### Generation backend (batched transformers; vLLM optional)

`pop rag` batches generation on the GPU and picks its backend automatically: **vLLM if it's
installed and its engine initializes**, otherwise a **batched `transformers`** pipeline --
still on the GPU, many samples at once, so far faster than the old one-at-a-time path (and
resumable either way).

This notebook does **not** install vLLM by default. The current Colab image ships CUDA 12
runtime libs, while `pip install vllm` pulls a CUDA-13 wheel that fails to load
(`libcudart.so.13: cannot open shared object file`) and can disturb the preinstalled torch.
Batched transformers on Colab's working torch is the dependable path. **To try vLLM anyway**,
add a cell with `!pip install vllm` before the sweep: if its engine initializes the code uses
it, and if not it falls back to transformers automatically -- no crash.


### Optional: Weights & Biases

The RAG sweep does no training, so W&B is not used on the Run-all path at all -- the file
logs + `STATUS.md` cover progress tracking. If you want a W&B run anyway, add a new cell with
`import wandb; wandb.login()` and paste **your own** key when prompted -- the key is never
stored in this notebook or the repo. It is deliberately not a default cell so that **Run all**
never stalls waiting for input.

In [ ]:
# Quick CPU-only sanity check: print the 16-step plan and run preflight without touching a GPU.
!python scripts/run_rag.py --list

In [ ]:
# The long-running cell. Re-running after any interruption continues where it left off:
# each config's `rag` (predictions.jsonl) and `eval` (results/*.json) step is skipped once
# its artifact exists.
!python scripts/run_rag.py

In [ ]:
import json
from pathlib import Path

status = Path("logs/rag/sweep/STATUS.md")
if status.exists():
    print(status.read_text(encoding="utf-8"))
summary = Path("logs/rag/sweep/SUMMARY.md")
if summary.exists():
    print(summary.read_text(encoding="utf-8"))
for path in sorted(Path("results").glob("rag_*_test.json")):
    metrics = json.loads(path.read_text(encoding="utf-8"))["metrics"]
    print(
        f"{path.name}: CodeBLEU={metrics['codebleu']:.4f} "
        f"syntax={metrics['syntax_valid_rate']:.4f} EM={metrics['em']:.4f} n={metrics['n']}"
    )

### If the session disconnects or hits the GPU quota

Normal and expected. Reopen this notebook and **Run all** again: finished `rag`/`eval`
steps are skipped instantly. A config that was only **partway** through when the session
dropped also resumes -- each config checkpoints its predictions to a
`predictions.jsonl.partial` file on Drive and only finalizes to `predictions.jsonl` when
complete, so the re-run continues at the first unfinished sample, not the first unfinished
config. Progress at any time: `Drive/pop_cycle3/logs/rag/sweep/STATUS.md`.
